# 02 · Stage 2: curve panel and curve pricer (spec §5)

**Gate:** the curve acceptance test. On 15 Mar 2016 the residuals reproduce the Module 2 targets: 1½% 2026 +2.70bp, 4¼% 2055 −1.91bp, 4½% 2034 +1.38bp, and a mean absolute residual of 2.6bp across 39 gilts.

**What Stage 2 delivers**
- `gilt_study.curve`, rewritten:
  - `build_panel()` / `load_panel()`: one row per BoE curve date from 1 Jan 1998 to 15 Oct 2026. Output: `data/processed_data/curves/boe_nominal_spot_daily.csv.gz`.
  - `Curve`: one day's curve with the spec §5 rules.
  - `curve_on()`: the curve for a date, falling back to the previous date (flagged) if it is missing.
  - `model_dirty()`, `model_yield()`, `beyond_curve()`.
- `exercises/exercise_b.py`: the acceptance test, now on the panel, with first issue and first coupon dates from `gilt_terms` (Stage 1, P3–P4).
- `exercises/curve_checks.py`: two checks of the extrapolation beyond the curve's last maturity (sections 4–5).

**Conventions (spec §5).** None of these is new; each comes from the spec or a logged decision.

| Item | Rule | Source |
|---|---|---|
| Rates | BoE spot z, % per year, continuously compounded; DF = exp(−z/100 · t), t = days / 365.25 | spec §5 |
| Splice | short-end sheet ≤ 5y (monthly grid) + long sheet > 5y (half-yearly) | spec §5 |
| Interpolation | natural cubic spline in z | spec §5 |
| Below the first maturity | z flat | spec §5 |
| Beyond the last maturity T | **flat forward**: hold the BoE instantaneous forward f(T) from the forward sheet, so z(t)·t = z(T)·T + f(T)·(t − T). Flag every price that uses it | spec §5, decision #5 |
| Missing curve date | previous available date, flagged | spec §5 |
| When-issued | a gilt settling before its first issue date is discounted from the first issue date | chat decision 19 Oct (proposed log entry #10) |
| Model yield | semi-annual GRY at which `gilt_price()` reproduces the curve dirty price; one `cash_flows()` feeds both | spec §4–5 |

In [1]:
import datetime as dt
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns

from gilt_study import config, curve, paths, plotting
from gilt_study.exercises import curve_checks, exercise_b
from gilt_study.pricing import is_business_day

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)
plotting.set_seaborn_theme()


def fmt(df, formats):
    # copy of df with the given columns formatted as strings (no jinja2 needed)
    out = df.copy()
    for col, f in formats.items():
        out[col] = out[col].map(lambda v: f.format(v) if pd.notna(v) else "")
    return out


P = curve.load_panel()          # builds from raw_data in ~20 s if the file is missing
mats = curve.maturity_columns(P)
print(f"panel: {len(P):,} curve dates, {P.index[0].date()} to {P.index[-1].date()}; {len(mats)} maturities "
      f"({mats[0]:.4f}y to {mats[-1]:.1f}y)")

panel: 7,274 curve dates, 1998-01-02 to 2026-10-15; 130 maturities (0.0833y to 40.0y)


## 1. The panel: every business day, and how far each curve reaches

In [2]:
days = pd.date_range(P.index[0], P.index[-1], freq="D")
bdays = {d for d in days if is_business_day(d.date())}
print("UK business days with no curve:", len(bdays - set(P.index)),
      "| curve rows on weekends or UK holidays:", len(set(P.index) - bdays))
reach = (P.groupby(P.index.year)
         .agg(curve_dates=("t_max", "size"), shortest_min=("t_min", "min"), shortest_max=("t_min", "max"),
              longest_min=("t_max", "min"), longest_max=("t_max", "max"),
              f_at_longest_mean=("f_tmax", "mean"), z_at_longest_mean=("z_tmax", "mean")))
reach.index.name = "year"
reach.round(3)

UK business days with no curve: 0 | curve rows on weekends or UK holidays: 0


,curve_dates,shortest_min,shortest_max,longest_min,longest_max,f_at_longest_mean,z_at_longest_mean
year,,,,,,,
1998,253,0.083,0.583,23.0,25.0,4.493,5.213
1999,252,0.083,0.583,25.0,25.0,3.770,4.368
2000,252,0.083,0.417,25.0,25.0,3.596,4.240
2001,253,0.083,0.083,25.0,25.0,3.881,4.548
2002,252,0.083,0.083,25.0,25.0,4.003,4.631
2003,253,0.083,0.083,25.0,25.0,4.351,4.600
2004,254,0.083,0.083,25.0,25.0,4.155,4.597
2005,252,0.083,0.083,25.0,25.0,4.050,4.288
2006,252,0.083,0.083,25.0,25.0,3.380,4.028


**Reading the table.**
- **No UK business day is missing** from 2 Jan 1998 to 15 Oct 2026, and no curve falls on a holiday. So the missing-date fallback is coded and tested (`test_missing_date_uses_previous_curve_and_flags`) but never fires on a business day.
- The curve reaches **25 years from 29 Jan 1998 to end-2015**, and **40 years from 2016**. In January 1998 it reaches only 23 years, but that is before the sample starts.
- **The forward at the last maturity is usually below the spot rate there**: on 71% of days before 2016 and on every day since (compare the last two columns). The BoE's fitted forward curve bends down at its long end, so a flat-forward extrapolation continues the curve *downwards* beyond T. Flat z would hold it level. Sections 3–5 show what that does to prices.

## 2. What the curves look like, and the extrapolated region

Five dates across the sample. The solid line is the BoE curve (spline through the published points). The dashed line is the flat-forward extension to 55 years. The dots mark the last published maturity.

In [3]:
snap_dates = [dt.date(1998, 11, 2), dt.date(2008, 12, 31), dt.date(2016, 3, 15),
              dt.date(2022, 9, 27), dt.date(2026, 9, 30)]
cmap = plotting.colour_map([f"{d:%d %b %Y}" for d in snap_dates])
grid = np.linspace(0.05, 55, 600)
rows = []
for d in snap_dates:
    c, used, fb = curve.curve_on(P, d)
    for t in grid:
        rows.append(dict(date=f"{d:%d %b %Y}", years=t, z=float(c(t)), region="BoE" if t <= c.t_max else "extrapolated"))
S = pd.DataFrame(rows)

fig_s, ax = plt.subplots(figsize=(9.5, 5))
for label, colour in cmap.items():
    s = S[S.date == label]
    a, b = s[s.region == "BoE"], s[s.years >= s.loc[s.region == "BoE", "years"].max()]
    ax.plot(a.years, a.z, color=colour, lw=2, label=label)
    ax.plot(b.years, b.z, color=colour, lw=2, ls="--")
    ax.scatter(a.years.iloc[-1], a.z.iloc[-1], color=colour, s=30, zorder=3)
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Spot rate z (%, cont. comp.)")
ax.set_title("BoE nominal spot curves; dashed = flat-forward extrapolation")
ax.legend(loc="lower right", ncols=2)
fig_s.tight_layout()

fig_p = go.Figure()
for label, colour in cmap.items():
    s = S[S.date == label]
    a, b = s[s.region == "BoE"], s[s.years >= s.loc[s.region == "BoE", "years"].max()]
    fig_p.add_trace(go.Scatter(x=a.years, y=a.z, mode="lines", name=label, legendgroup=label,
                               line=dict(color=colour, width=2),
                               hovertemplate=label + "<br>%{x:.1f}y: %{y:.3f}%<extra>BoE</extra>"))
    fig_p.add_trace(go.Scatter(x=b.years, y=b.z, mode="lines", name=label + " (extrapolated)", legendgroup=label,
                               showlegend=False, line=dict(color=colour, width=2, dash="dash"),
                               hovertemplate=label + "<br>%{x:.1f}y: %{y:.3f}%<extra>flat forward</extra>"))
fig_p.update_layout(template=plotting.PLOTLY_TEMPLATE, height=480,
                    title="BoE nominal spot curves; dashed = flat-forward extrapolation",
                    xaxis_title="Maturity (years)", yaxis_title="Spot rate z (%, cont. comp.)")
plotting.save_pair("curve_snapshots", fig_s, fig_p)
fig_p.show()

## 3. The acceptance test: 15 Mar 2016

Settlement is 16 Mar 2016. All 39 single-dated conventional gilts with more than a year to maturity are priced off the curve, and each model yield is compared with the DMO reference yield. The residual is DMO yield − model yield, so **positive means the gilt is cheap to the curve**.

In [4]:
R = exercise_b.residuals(exercise_b.DATE, P)
tgt = R[R.gilt.isin(exercise_b.TARGETS_BP)].assign(target_bp=lambda d: d.gilt.map(exercise_b.TARGETS_BP))
print(tgt[["gilt", "years", "dmo_yield", "model_yield", "residual_bp", "target_bp"]].to_string(index=False))
print(f"\ngilts: {len(R)} (target {exercise_b.TARGET_COUNT})")
print(f"mean |residual|, flat forward (study):   {R.residual_bp.abs().mean():.2f} bp")
print(f"mean |residual|, flat z (Module 2):       {R.residual_bp_flat_z.abs().mean():.2f} bp   <- target 2.6")
print(f"mean |residual|, the 36 within the curve: {R.loc[~R.beyond_curve, 'residual_bp'].abs().mean():.2f} bp")
R[R.beyond_curve][["gilt", "years", "dmo_yield", "model_yield", "residual_bp", "model_yield_flat_z", "residual_bp_flat_z"]]

                  gilt  years  dmo_yield  model_yield  residual_bp  target_bp
1½% Treasury Gilt 2026  10.35   1.647660       1.6207         2.70       2.70
4½% Treasury Gilt 2034  18.48   2.133352       2.1195         1.38       1.38
4¼% Treasury Gilt 2055  39.73   2.162618       2.1817        -1.91      -1.91

gilts: 39 (target 39)
mean |residual|, flat forward (study):   2.84 bp
mean |residual|, flat z (Module 2):       2.64 bp   <- target 2.6
mean |residual|, the 36 within the curve: 2.70 bp


,gilt,years,dmo_yield,model_yield,residual_bp,model_yield_flat_z,residual_bp_flat_z
36,4% Treasury Gilt 2060,43.85,2.157037,2.1506,0.65,2.1799,-2.29
37,2½% Treasury Gilt 2065,49.35,2.158993,2.1032,5.58,2.1807,-2.17
38,3½% Treasury Gilt 2068,52.35,2.167949,2.0949,7.31,2.1804,-1.25


In [5]:
fig_s, fig_p = exercise_b.charts(R)
plotting.save_pair(exercise_b.CHART, fig_s, fig_p)
fig_p.show()

**Result.**
- **All three targets reproduce exactly** (+2.70, −1.91, +1.38 bp), and the gilt count is 39. These three gilts are inside the 40-year curve, so the extrapolation choice cannot affect them.
- **The mean absolute residual is 2.84 bp with flat forward, against the 2.6 bp target.** Module 2's 2.64 bp is reproduced exactly with flat z. The whole difference comes from the **three gilts beyond 40 years** (2060, 2065, 2068). Under flat forward they move from −2.3/−2.2/−1.3 bp to +0.7/+5.6/+7.3 bp.
- The reason is in the next cell. On this date the BoE forward at 40 years is about **1.5%**, while the 40-year spot rate is about **2.17%**. Holding that low forward constant pulls the 50-year spot rate down to about 2.04%. The market priced the ultra-longs flat, at around 2.16%.

The test as written in spec §5 was calibrated on flat z (Module 2). **This needs your call at the gate** (see section 6).

In [6]:
fwd = curve._read_sheet(pd.ExcelFile(paths.curve_workbook("2016 to 2024")), 2).loc["2016-03-15"]
spot = P.loc["2016-03-15", mats]
near = [20.0, 25.0, 30.0, 35.0, 37.5, 39.0, 40.0]
pd.DataFrame({"spot z (%)": spot[near].values, "instantaneous forward (%)": fwd[near].values},
             index=pd.Index(near, name="maturity (y)")).round(3)

,spot z (%),instantaneous forward (%)
maturity (y),,
20.0,2.359,3.099
25.0,2.453,2.507
30.0,2.401,1.796
35.0,2.280,1.396
37.5,2.221,1.389
39.0,2.189,1.441
40.0,2.171,1.495


## 4. Check 1: gilts beyond the curve, 2002–2015 (DMO yields)

Before 2016 the curve stops at 25 years, so every gilt with more than 25 years to run is priced partly on the extrapolation. The DMO reference yields let us measure how well each extrapolation prices those gilts. The sample is the last reference-price date of each month, from 25 Nov 2002 to 21 Jul 2017: every single-dated gilt with known terms and more than a year to run, excluding WI lines.

In [7]:
M = pd.read_csv(curve_checks.OUT_DIR / "monthly_residuals.csv", parse_dates=["date"])
M["group"] = np.where(M.beyond_curve, "beyond the curve (25y)", "within the curve")
print("2016-17 (curve to 40y): gilt-days beyond the curve:", int(M.loc[M.date >= "2016-01-01", "beyond_curve"].sum()))
pre_ = M[M.date < "2016-01-01"]
tab = (pre_.groupby("group").agg(gilt_days=("gilt", "size"), dates=("date", "nunique"),
                              mean_bp_flat_forward=("residual_bp", "mean"),
                              mean_abs_bp_flat_forward=("residual_bp", lambda s: s.abs().mean()),
                              mean_bp_flat_z=("residual_bp_flat_z", "mean"),
                              mean_abs_bp_flat_z=("residual_bp_flat_z", lambda s: s.abs().mean())).round(2))
tab

2016-17 (curve to 40y): gilt-days beyond the curve: 63


,gilt_days,dates,mean_bp_flat_forward,mean_abs_bp_flat_forward,mean_bp_flat_z,mean_abs_bp_flat_z
group,,,,,,
beyond the curve (25y),1073,158,-4.28,4.71,-3.71,6.82
within the curve,3741,158,-0.32,3.05,-0.32,3.05


In [8]:
edges = [1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 55]
pre = M[M.date < "2016-01-01"].copy()
pre["bucket"] = pd.cut(pre.years, edges, labels=[f"{a}–{b}" for a, b in zip(edges[:-1], edges[1:])])
long_ = (pre.melt(id_vars=["bucket"], value_vars=["residual_bp", "residual_bp_flat_z"], var_name="method", value_name="bp")
         .replace({"method": {"residual_bp": "flat forward (study)", "residual_bp_flat_z": "flat z"}}))
agg = long_.groupby(["bucket", "method"], observed=True)["bp"].agg(mean="mean", n="size").reset_index()
cm = plotting.colour_map(["flat forward (study)", "flat z"])
title = "Mean residual by years to maturity, 2002–2015 (curve ends at 25y)"

fig_s, ax = plt.subplots(figsize=(9.5, 4.6))
ax.axhline(0, color=plotting.BASELINE, lw=1)
ax.axvspan(4.5, 9.5, color=plotting.NEUTRAL_BAND, zorder=0)
sns.pointplot(data=agg, x="bucket", y="mean", hue="method", palette=cm, dodge=0.25, markers="o",
              linestyles="-", ax=ax)
ax.text(7, ax.get_ylim()[1] * 0.9, "beyond 25y: extrapolated", ha="center", color=plotting.INK_SECONDARY, fontsize=9)
ax.set_xlabel("Years to maturity")
ax.set_ylabel("Mean DMO yield − model yield (bp)")
ax.set_title(title)
ax.legend(title=None)
fig_s.tight_layout()

fig_p = go.Figure()
fig_p.add_vrect(x0=4.5, x1=9.5, fillcolor=plotting.NEUTRAL_BAND, line_width=0, layer="below",
                annotation_text="beyond 25y: extrapolated", annotation_position="top")
fig_p.add_hline(y=0, line_color=plotting.BASELINE, line_width=1)
for m, colour in cm.items():
    a = agg[agg.method == m]
    fig_p.add_trace(go.Scatter(x=list(range(len(a))), y=a["mean"], mode="lines+markers", name=m,
                               line=dict(color=colour), customdata=np.stack([a.bucket.astype(str), a.n], axis=1),
                               hovertemplate="%{customdata[0]}y: %{y:+.2f} bp<br>%{customdata[1]} gilt-days<extra>" + m + "</extra>"))
fig_p.update_layout(template=plotting.PLOTLY_TEMPLATE, height=440, title=title,
                    xaxis=dict(title="Years to maturity", tickvals=list(range(len(edges) - 1)),
                               ticktext=[f"{a}–{b}" for a, b in zip(edges[:-1], edges[1:])]),
                    yaxis_title="Mean DMO yield − model yield (bp)", legend=dict(orientation="h", y=-0.2))
plotting.save_pair("curve_residuals_by_maturity", fig_s, fig_p)
fig_p.show()

**Reading it.**
- **Within the curve**, the mean residual in each maturity bucket is within ±1 bp of zero. The mean absolute residual is 3.0 bp, falling from about 4 bp at 1–5 years to 1.5 bp at 20–25 years.
- **Beyond 25 years**, both extrapolations leave gilts looking **rich**: the model yield is above the DMO yield. The gap widens with maturity. With flat forward it is about 0.6 bp at 25–30 years, 2 bp at 30–35, 6 bp at 35–40 and 10–11 bp beyond 40.
- **Flat forward has the smaller absolute error overall** (see the table), but neither removes the bias. The exception is 40–45 years, where flat z is slightly closer. This bias is exactly what decision #5 worries about. For a long auction before 2016, M1 (AAY − model yield) would understate the concession by a few bp. M1b nets out each gilt's own pre-auction residual, so it removes this level bias by construction.

## 5. Check 2: cut the 2016–2026 curve at 25 years

From 2016 the BoE publishes the curve to 40 years. Cutting each month-end curve at 25 years and extrapolating it tests both methods against the BoE's own longer curve. Flat forward uses the BoE forward at 25 years. The test covers gilts with 25–40 years to run, from Jan 2016 to Oct 2026, and gives the error in model yield (extrapolated − full curve).

In [9]:
X = pd.read_csv(curve_checks.OUT_DIR / "extrapolation_truncation.csv", parse_dates=["date"])
X["beyond"] = pd.cut(X.years_beyond, [0, 5, 10, 15], labels=["25–30y", "30–35y", "35–40y"])
print(X.groupby("beyond", observed=True).agg(
    gilt_days=("gilt", "size"),
    mean_bp_flat_forward=("error_bp_flat_forward", "mean"), mean_abs_bp_flat_forward=("error_bp_flat_forward", lambda s: s.abs().mean()),
    mean_bp_flat_z=("error_bp_flat_z", "mean"), mean_abs_bp_flat_z=("error_bp_flat_z", lambda s: s.abs().mean())).round(2).to_string())
print(f"\nshare of gilt-days where flat forward is closer: {(X.error_bp_flat_forward.abs() < X.error_bp_flat_z.abs()).mean():.1%}")

        gilt_days  mean_bp_flat_forward  mean_abs_bp_flat_forward  mean_bp_flat_z  mean_abs_bp_flat_z
beyond                                                                                               
25–30y        670                  1.01                      1.01            1.46                1.65
30–35y        377                  4.79                      4.79            6.51                6.53
35–40y        316                 12.05                     12.05           13.69               13.69

share of gilt-days where flat forward is closer: 64.5%


In [10]:
X["years_beyond_bin"] = (X.years_beyond // 1).astype(int) + 0.5
lx = (X.melt(id_vars=["years_beyond_bin"], value_vars=["error_bp_flat_forward", "error_bp_flat_z"],
             var_name="method", value_name="bp")
      .replace({"method": {"error_bp_flat_forward": "flat forward (study)", "error_bp_flat_z": "flat z"}}))
ag = lx.groupby(["years_beyond_bin", "method"])["bp"].agg(mean="mean", p10=lambda s: s.quantile(.1),
                                                          p90=lambda s: s.quantile(.9), n="size").reset_index()
title = "Extrapolation error when the 2016–2026 curve is cut at 25 years"
ylab = "Model yield: extrapolated − full curve (bp)"

fig_s, ax = plt.subplots(figsize=(9.5, 4.6))
ax.axhline(0, color=plotting.BASELINE, lw=1)
for m, colour in cm.items():
    a = ag[ag.method == m]
    ax.fill_between(a.years_beyond_bin, a.p10, a.p90, color=colour, alpha=0.15, lw=0)
    ax.plot(a.years_beyond_bin, a["mean"], color=colour, label=m + " (mean; band = 10th–90th pct)")
ax.set_xlabel("Years beyond the 25-year cut")
ax.set_ylabel(ylab)
ax.set_title(title)
ax.legend(loc="upper left")
fig_s.tight_layout()

fig_p = go.Figure()
fig_p.add_hline(y=0, line_color=plotting.BASELINE, line_width=1)
for m, colour in cm.items():
    a = ag[ag.method == m]
    r, g, b = (int(colour[i:i + 2], 16) for i in (1, 3, 5))
    fig_p.add_trace(go.Scatter(x=np.r_[a.years_beyond_bin, a.years_beyond_bin[::-1]], y=np.r_[a.p90, a.p10[::-1]],
                               fill="toself", fillcolor=f"rgba({r},{g},{b},0.15)", line=dict(width=0),
                               hoverinfo="skip", showlegend=False, legendgroup=m))
    fig_p.add_trace(go.Scatter(x=a.years_beyond_bin, y=a["mean"], mode="lines+markers", name=m, legendgroup=m,
                               line=dict(color=colour), customdata=np.stack([a.p10, a.p90, a.n], axis=1),
                               hovertemplate=("%{x:.1f}y beyond: mean %{y:+.2f} bp<br>10th–90th pct "
                                              "%{customdata[0]:+.1f} to %{customdata[1]:+.1f}<br>%{customdata[2]} gilt-days"
                                              "<extra>" + m + "</extra>")))
fig_p.update_layout(template=plotting.PLOTLY_TEMPLATE, height=440, title=title,
                    xaxis_title="Years beyond the 25-year cut", yaxis_title=ylab, legend=dict(orientation="h", y=-0.2))
plotting.save_pair("curve_truncation_test", fig_s, fig_p)
fig_p.show()

**Reading it.**
- **Both methods overstate long yields when the curve stops at 25 years.** This is the same sign as the 2002–2015 residuals in section 4: gilts beyond the curve look rich.
- The error grows with distance: about **1 bp** for 25–30 years, **5 bp** for 30–35 years and **12 bp** for 35–40 years with flat forward. Flat z is worse in every bucket.
- Flat forward is closer on 65% of gilt-days.
- So, against both the DMO prices (section 4) and the BoE's own longer curve (here), **flat forward is the better of the two, which supports decision #5**. But the error is still large for the longest pre-2016 auctions. That makes the flag and the robustness checks in spec §5 essential: drop flagged auctions, and compare with M1b.
- **15 Mar 2016 (section 3) goes the other way.** Extrapolating beyond 40 years from a steeply falling forward makes ultra-longs look *cheap*. The direction of the error depends on the slope of the BoE forward curve at T.

## 6. Stopping here: the Stage 2 gate

**Tests run** (`pytest`): see the README Stage 2 note. They cover the extrapolation maths, the when-issued rule, the missing-date fallback, panel coverage, panel rows identical to the raw workbook, and the acceptance test.

**Decisions for you**

1. **Acceptance test, mean residual.** With the study's flat-forward extrapolation the three targets and the count reproduce exactly, but the mean absolute residual is **2.84 bp** against the 2.6 bp target. Flat z reproduces Module 2's 2.64 bp; within the 40-year curve it is 2.70 bp. The difference is entirely the three gilts beyond 40 years. Options:
   - (a) accept the gate on the three targets, the count and the flat-z mean (2.64), and record that flat forward moves the three ultra-longs;
   - (b) re-state the §5 mean target as "within the curve" (2.70 bp);
   - (c) revisit decision #5.

   Sections 4–5 favour flat forward over flat z overall, so (a) or (b) keeps the decision.
2. **Extrapolation flag.** As specified, an auction is flagged if *any* cash flow lies beyond the curve. Before 2016 that is every gilt over 25 years. Sections 4–5 show the error is about 1 bp at 25–30 years but 5–12 bp at 30–40 years. Keep the single flag (spec), or also record *years beyond the curve* in the auction panel, so the robustness check can use a threshold? Recording the number changes nothing by default.